# Funnel Analysis — Edge Cases & Real-World Intricacies

**Author:**Ahmad Umer
**Date:** 28/4/2026  
**Tools:** Python, Pandas, NumPy

A production-grade funnel analysis simulation exposing four real-world
data intricacies invisible to standard analytics tools.



## Stage 1 — Generating the Spaghetti Dataset

Simulating a real e-commerce event log with four injected intricacies:
- Loopbacks
- Duplicate events
- Cross-device fragmentation
- Orphaned purchases

In [12]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# ============================================================
# CONFIGURATION
# ============================================================
np.random.seed(42)
random.seed(42)

N_USERS          = 500       # total simulated users
FUNNEL_STEPS     = ['home', 'product_view', 'add_to_cart', 'checkout_click', 'purchase']
DEVICES          = ['Mobile_App', 'Web_Desktop', 'Tablet']
BASE_TIME        = datetime(2024, 1, 1, 9, 0, 0)

events = []  # we'll collect all raw event dicts here

# ============================================================
# HELPER: generate a plausible timestamp offset (seconds)
# ============================================================
def next_ts(current_ts, min_sec=30, max_sec=600):
    """Advance time by a random interval — simulates real user pacing."""
    return current_ts + timedelta(seconds=random.randint(min_sec, max_sec))

# ============================================================
# MAIN SIMULATION LOOP
# ============================================================
for user_id in range(1, N_USERS + 1):

    uid = f"user_{user_id:04d}"
    session_start = BASE_TIME + timedelta(
        hours=random.randint(0, 72),
        minutes=random.randint(0, 59)
    )
    primary_device = random.choice(DEVICES)
    ts = session_start

    # ----------------------------------------------------------
    # INTRICACY #3 SETUP: Cross-Device users (≈15% of users)
    # These users will SWITCH devices mid-journey.
    # ----------------------------------------------------------
    is_cross_device = random.random() < 0.15
    switch_point    = random.choice(['add_to_cart', 'checkout_click'])  # where they switch
    secondary_device = 'Web_Desktop' if primary_device == 'Mobile_App' else 'Mobile_App'

    # ----------------------------------------------------------
    # INTRICACY #4: Orphaned Purchase — no cart, no checkout
    # ≈3% of users — simulates a broken tracking pipeline or
    # server-side purchase event firing without front-end steps.
    # ----------------------------------------------------------
    if random.random() < 0.03:
        ts = next_ts(ts)
        events.append({
            'user_id':    uid,
            'event':      'purchase',
            'timestamp':  ts,
            'device':     primary_device,
            'session_id': f"{uid}_orphan"
        })
        continue  # skip to next user — nothing else fires for them

    # ----------------------------------------------------------
    # NORMAL FUNNEL — with intricacies injected step by step
    # ----------------------------------------------------------
    session_id = f"{uid}_s1"
    reached_cart = False

    for step_idx, step in enumerate(FUNNEL_STEPS):

        # ── Drop-off logic: not every user completes every step ──
        # Drop probability increases deeper in the funnel
        drop_prob = [0.05, 0.15, 0.25, 0.35, 0.30][step_idx]
        if random.random() < drop_prob:
            break   # user dropped off — stop recording events for them

        # ── Advance timestamp realistically ──
        ts = next_ts(ts)

        # ── Device switching for cross-device users ──
        current_device = primary_device
        if is_cross_device and step == switch_point:
            # Simulate the user picking up a different device
            # Gap of several hours — common "research on phone, buy on laptop" pattern
            ts = ts + timedelta(hours=random.randint(4, 12))
            current_device = secondary_device
            session_id = f"{uid}_s2"  # new session on new device

        # ── Log the event ──
        events.append({
            'user_id':    uid,
            'event':      step,
            'timestamp':  ts,
            'device':     current_device,
            'session_id': session_id
        })

        if step == 'add_to_cart':
            reached_cart = True

        # ── INTRICACY #1: Loopback — after cart, some users go back to home ──
        # ≈20% of users who reached cart bounce back and revisit product pages
        if step == 'add_to_cart' and random.random() < 0.20:
            # They go: cart → home → product_view → back to cart
            for loopback_event in ['home', 'product_view', 'add_to_cart']:
                ts = next_ts(ts, min_sec=10, max_sec=120)
                events.append({
                    'user_id':    uid,
                    'event':      loopback_event,
                    'timestamp':  ts,
                    'device':     current_device,
                    'session_id': session_id
                })

        # ── INTRICACY #2: Duplicate Events — checkout_click fires 3× rapidly ──
        # Simulates a double-tap, React re-render race condition, or flaky SDK
        if step == 'checkout_click' and random.random() < 0.25:
            for _ in range(2):  # fire 2 extra times = 3 total
                ts = ts + timedelta(seconds=random.randint(1, 2))  # <2 sec gap — that's the tell
                events.append({
                    'user_id':    uid,
                    'event':      'checkout_click',
                    'timestamp':  ts,
                    'device':     current_device,
                    'session_id': session_id
                })

# ============================================================
# ASSEMBLE THE DATAFRAME
# ============================================================
df = pd.DataFrame(events)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)

# ============================================================
# QUICK SANITY REPORT — run this immediately after generating
# ============================================================
print("=" * 55)
print("  RAW EVENT LOG — SANITY REPORT")
print("=" * 55)
print(f"  Total raw events        : {len(df):,}")
print(f"  Unique users            : {df['user_id'].nunique():,}")
print(f"  Unique sessions         : {df['session_id'].nunique():,}")
print(f"  Devices seen            : {df['device'].unique().tolist()}")
print(f"  Date range              : {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"  Event type counts:")
print(df['event'].value_counts().to_string())
print("=" * 55)

print("\n--- df.head(10) ---")
print(df.head(10).to_string())

print("\n--- df.dtypes ---")
print(df.dtypes)

  RAW EVENT LOG — SANITY REPORT
  Total raw events        : 1,711
  Unique users            : 468
  Unique sessions         : 502
  Devices seen            : ['Tablet', 'Mobile_App', 'Web_Desktop']
  Date range              : 2024-01-01 09:28:47 → 2024-01-04 14:44:32
  Event type counts:
event
home              512
product_view      434
add_to_cart       337
checkout_click    274
purchase          154

--- df.head(10) ---
     user_id           event           timestamp      device    session_id
0  user_0001            home 2024-01-01 23:10:48      Tablet  user_0001_s1
1  user_0003            home 2024-01-02 10:45:36      Tablet  user_0003_s1
2  user_0003    product_view 2024-01-02 10:48:49      Tablet  user_0003_s1
3  user_0003     add_to_cart 2024-01-02 10:55:07      Tablet  user_0003_s1
4  user_0004            home 2024-01-03 04:07:14  Mobile_App  user_0004_s1
5  user_0004    product_view 2024-01-03 04:16:53  Mobile_App  user_0004_s1
6  user_0005            home 2024-01-03 09:06:16 

## Stage 2 — Naive Funnel vs Strict Sequence Funnel

Exposing how naive unique-user counting inflates conversion rates
and building a timestamp-enforced strict sequence funnel.

In [13]:
FUNNEL_STEPS = ['home', 'product_view', 'add_to_cart', 'checkout_click', 'purchase']

# ============================================================
# NAIVE FUNNEL — Unique users per step, no sequence enforcement
# ============================================================
naive_funnel = []

for step in FUNNEL_STEPS:
    unique_users = df[df['event'] == step]['user_id'].nunique()
    naive_funnel.append({'step': step, 'unique_users': unique_users})

naive_df = pd.DataFrame(naive_funnel)

naive_df['prev_users'] = naive_df['unique_users'].shift(1)
naive_df['conversion_rate'] = (
    naive_df['unique_users'] / naive_df['prev_users']
).round(3)


top_of_funnel = naive_df.loc[naive_df['step'] == 'home', 'unique_users'].values[0]
naive_df['overall_conv'] = (naive_df['unique_users'] / top_of_funnel).round(3)

print("=" * 60)
print("  NAIVE FUNNEL — No Sequence Enforcement")
print("=" * 60)
print(naive_df[['step','unique_users','conversion_rate','overall_conv']].to_string(index=False))
print("=" * 60)

  NAIVE FUNNEL — No Sequence Enforcement
          step  unique_users  conversion_rate  overall_conv
          home           455              NaN         1.000
  product_view           377            0.829         0.829
   add_to_cart           280            0.743         0.615
checkout_click           184            0.657         0.404
      purchase           154            0.837         0.338


In [14]:
# ============================================================
# STEP 1: Get the first timestamp per user per event
# ============================================================
first_touch = (
    df[df['event'].isin(FUNNEL_STEPS)]
    .groupby(['user_id', 'event'])['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp': 'first_ts'})
)

# ============================================================
# STEP 2: Pivot to wide format — one row per user
# ============================================================
pivot_df = first_touch.pivot_table(
    index='user_id',
    columns='event',
    values='first_ts',
    aggfunc='min'
)
pivot_df = pivot_df[FUNNEL_STEPS]
pivot_df.columns.name = None

# ============================================================
# STEP 3: Walk the funnel enforcing strict sequence
# ============================================================
strict_funnel = []
qualifying_pivot = pivot_df.copy()

for i, step in enumerate(FUNNEL_STEPS):
    if i == 0:
        mask = qualifying_pivot[step].notna()
    else:
        prev_step = FUNNEL_STEPS[i - 1]
        mask = (
            qualifying_pivot[step].notna() &
            (qualifying_pivot[step] > qualifying_pivot[prev_step])
        )
    qualifying_pivot = qualifying_pivot[mask]
    strict_funnel.append({
        'step':         step,
        'strict_users': len(qualifying_pivot)
    })

strict_df = pd.DataFrame(strict_funnel)
strict_df['prev_users']      = strict_df['strict_users'].shift(1)
strict_df['conversion_rate'] = (
    strict_df['strict_users'] / strict_df['prev_users']
).round(3)
top = strict_df.loc[0, 'strict_users']
strict_df['overall_conv'] = (strict_df['strict_users'] / top).round(3)

# ============================================================
# STEP 4: Side-by-side comparison — this is your finding
# ============================================================

# Rebuild naive for comparison
naive_funnel = []
for step in FUNNEL_STEPS:
    naive_funnel.append({
        'step': step,
        'naive_users': df[df['event'] == step]['user_id'].nunique()
    })
naive_df = pd.DataFrame(naive_funnel)

# Merge both
comparison = naive_df.merge(
    strict_df[['step', 'strict_users', 'conversion_rate', 'overall_conv']],
    on='step'
)
comparison['user_gap'] = comparison['naive_users'] - comparison['strict_users']
comparison['gap_pct']  = (
    comparison['user_gap'] / comparison['naive_users'] * 100
).round(1)

print("=" * 75)
print("  NAIVE vs STRICT — The Gap Is Your Finding")
print("=" * 75)
print(comparison[['step','naive_users','strict_users',
                   'user_gap','gap_pct','conversion_rate','overall_conv']]
      .to_string(index=False))
print("=" * 75)

# ============================================================
# STEP 5: Identify the specific anomaly types
# ============================================================

# Find orphaned purchase users — bought without ever adding to cart
cart_users = set(df[df['event'] == 'add_to_cart']['user_id'])
purchase_users = set(df[df['event'] == 'purchase']['user_id'])
orphaned = purchase_users - cart_users

print(f"\n  Orphaned purchase users (no cart event): {len(orphaned)}")
print(f"  These users: {sorted(list(orphaned))[:10]} ...")

# Find cross-device users — more than one unique device per user
device_counts = (
    df.groupby('user_id')['device']
    .nunique()
    .reset_index()
    .rename(columns={'device': 'device_count'})
)
cross_device_users = device_counts[device_counts['device_count'] > 1]
print(f"\n  Cross-device users (2+ devices): {len(cross_device_users)}")

# Find duplicate checkout events — same user, same event, within 5 seconds
df_sorted = df.sort_values(['user_id', 'timestamp'])
df_sorted['prev_event'] = df_sorted.groupby('user_id')['event'].shift(1)
df_sorted['prev_ts']    = df_sorted.groupby('user_id')['timestamp'].shift(1)
df_sorted['time_diff_sec'] = (
    df_sorted['timestamp'] - df_sorted['prev_ts']
).dt.total_seconds()

duplicates = df_sorted[
    (df_sorted['event'] == 'checkout_click') &
    (df_sorted['prev_event'] == 'checkout_click') &
    (df_sorted['time_diff_sec'] <= 5)
]
print(f"\n  Duplicate checkout_click events (within 5 sec): {len(duplicates)}")
print("=" * 75)

  NAIVE vs STRICT — The Gap Is Your Finding
          step  naive_users  strict_users  user_gap  gap_pct  conversion_rate  overall_conv
          home          455           455         0      0.0              NaN         1.000
  product_view          377           377         0      0.0            0.829         0.829
   add_to_cart          280           280         0      0.0            0.743         0.615
checkout_click          184           184         0      0.0            0.657         0.404
      purchase          154           141        13      8.4            0.766         0.310

  Orphaned purchase users (no cart event): 13
  These users: ['user_0086', 'user_0121', 'user_0196', 'user_0212', 'user_0236', 'user_0286', 'user_0353', 'user_0373', 'user_0379', 'user_0420'] ...

  Cross-device users (2+ devices): 34

  Duplicate checkout_click events (within 5 sec): 90


## Stage 3 — Deduplication and Sessionization

Removing 90 duplicate checkout events and applying a 30-minute
inactivity timeout to correctly label session boundaries.



In [15]:
# ============================================================
# DEDUPLICATION — Remove duplicate events within a time window
# ============================================================

# Sort by user and time first — this is mandatory
df_sorted = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)

# For each row, look at the PREVIOUS row for the same user
# shift(1) moves values down by 1 row within each group
df_sorted['prev_event'] = (
    df_sorted.groupby('user_id')['event'].shift(1)
)
df_sorted['prev_ts'] = (
    df_sorted.groupby('user_id')['timestamp'].shift(1)
)

# Calculate time difference in seconds from previous event
df_sorted['time_diff_sec'] = (
    df_sorted['timestamp'] - df_sorted['prev_ts']
).dt.total_seconds()

print("--- Sample of shift() output ---")
print(df_sorted[['user_id','event','timestamp',
                  'prev_event','time_diff_sec']].head(15).to_string(index=False))
# A row is a duplicate if:
# 1. Same event as the previous event for this user
# 2. Within 5 seconds of the previous event

is_duplicate = (
    (df_sorted['event'] == df_sorted['prev_event']) &
    (df_sorted['time_diff_sec'] <= 5)
)

df_deduped = df_sorted[~is_duplicate].copy()

print("\n" + "=" * 55)
print("  DEDUPLICATION RESULTS")
print("=" * 55)
print(f"  Rows before dedup : {len(df_sorted):,}")
print(f"  Rows removed      : {is_duplicate.sum():,}")
print(f"  Rows after dedup  : {len(df_deduped):,}")
print(f"  Events removed    : ")
print(df_sorted[is_duplicate]['event'].value_counts().to_string())
print("=" * 55)

--- Sample of shift() output ---
  user_id          event           timestamp     prev_event  time_diff_sec
user_0001           home 2024-01-01 23:10:48            NaN            NaN
user_0003           home 2024-01-02 10:45:36            NaN            NaN
user_0003   product_view 2024-01-02 10:48:49           home          193.0
user_0003    add_to_cart 2024-01-02 10:55:07   product_view          378.0
user_0004           home 2024-01-03 04:07:14            NaN            NaN
user_0004   product_view 2024-01-03 04:16:53           home          579.0
user_0005           home 2024-01-03 09:06:16            NaN            NaN
user_0005   product_view 2024-01-03 09:11:42           home          326.0
user_0005    add_to_cart 2024-01-03 09:16:10   product_view          268.0
user_0005 checkout_click 2024-01-03 09:24:24    add_to_cart          494.0
user_0005       purchase 2024-01-03 09:31:13 checkout_click          409.0
user_0006           home 2024-01-03 06:17:40            NaN        

In [16]:
# ============================================================
# SESSIONIZATION — 30-minute inactivity timeout
# Applied to the DEDUPLICATED dataframe
# ============================================================

SESSION_TIMEOUT_MINUTES = 30

df_session = df_deduped.sort_values(
    ['user_id', 'timestamp']
).reset_index(drop=True)

# Step 1: Calculate gap from previous event per user
df_session['prev_ts_session'] = (
    df_session.groupby('user_id')['timestamp'].shift(1)
)
df_session['gap_minutes'] = (
    df_session['timestamp'] - df_session['prev_ts_session']
).dt.total_seconds() / 60

# Step 2: Flag new session boundaries
df_session['is_new_session'] = (
    df_session['gap_minutes'].isna() |
    (df_session['gap_minutes'] > SESSION_TIMEOUT_MINUTES)
)

# Step 3: Assign session numbers per user using cumsum
df_session['session_num'] = (
    df_session.groupby('user_id')['is_new_session']
    .cumsum()
    .astype(int)
)

# Step 4: Build clean readable session identifier
df_session['clean_session_id'] = (
    df_session['user_id'] + '_session_' +
    df_session['session_num'].astype(str)
)

# ============================================================
# RESULTS SUMMARY
# ============================================================
original_sessions  = df['session_id'].nunique()
new_sessions       = df_session['clean_session_id'].nunique()
sessions_created   = new_sessions - original_sessions

session_counts = (
    df_session.groupby('user_id')['clean_session_id']
    .nunique()
    .reset_index()
    .rename(columns={'clean_session_id': 'num_sessions'})
)
split_users = session_counts[session_counts['num_sessions'] > 1]

print("=" * 55)
print("  SESSIONIZATION RESULTS")
print("=" * 55)
print(f"  Events before sessionization : {len(df_deduped):,}")
print(f"  Events after sessionization  : {len(df_session):,}")
print(f"  Original session count       : {original_sessions:,}")
print(f"  New session count            : {new_sessions:,}")
print(f"  Net new sessions created     : {sessions_created:,}")
print(f"  Users with split sessions    : {len(split_users):,}")
print("=" * 55)

# ============================================================
# SAMPLE — Show one split session in full detail
# ============================================================
if len(split_users) > 0:
    sample_uid = split_users.iloc[0]['user_id']
    sample_data = df_session[df_session['user_id'] == sample_uid][
        ['user_id','event','timestamp',
         'gap_minutes','is_new_session','clean_session_id']
    ]
    print(f"\n  Sample split user — {sample_uid}")
    print(f"  (gap > {SESSION_TIMEOUT_MINUTES} min triggered a new session)\n")
    print(sample_data.to_string(index=False))

# ============================================================
# GAP DISTRIBUTION — understand your users idle behavior
# ============================================================
print("\n  Gap distribution across ALL events (minutes):")
print(df_session['gap_minutes'].describe().round(2).to_string())

print("\n  Events where gap exceeded 30 minutes:")
long_gaps = df_session[df_session['gap_minutes'] > 30]
print(f"  Count : {len(long_gaps):,}")
print(f"  These triggered {len(long_gaps):,} new session boundaries")

  SESSIONIZATION RESULTS
  Events before sessionization : 1,621
  Events after sessionization  : 1,621
  Original session count       : 502
  New session count            : 502
  Net new sessions created     : 0
  Users with split sessions    : 34

  Sample split user — user_0015
  (gap > 30 min triggered a new session)

  user_id          event           timestamp  gap_minutes  is_new_session    clean_session_id
user_0015           home 2024-01-03 20:57:21          NaN            True user_0015_session_1
user_0015   product_view 2024-01-03 21:01:05     3.733333           False user_0015_session_1
user_0015    add_to_cart 2024-01-04 07:03:58   602.883333            True user_0015_session_2
user_0015           home 2024-01-04 07:05:07     1.150000           False user_0015_session_2
user_0015   product_view 2024-01-04 07:05:48     0.683333           False user_0015_session_2
user_0015    add_to_cart 2024-01-04 07:06:07     0.316667           False user_0015_session_2
user_0015 checkout_

## Stage 4 — Cross-Device Attribution Analysis

Resolving cross-device fragmentation and calculating conversion credit
across First-Touch, Last-Touch, and Position-Based models.

In [17]:
# ============================================================
# STAGE 4 — CROSS-DEVICE ATTRIBUTION ANALYSIS
# ============================================================

# Step 1: Isolate the cross-device users we already identified
cross_device_uids = (
    df_session.groupby('user_id')['device']
    .nunique()
    .reset_index()
    .query('device > 1')
    ['user_id']
    .tolist()
)

print(f"Cross-device users confirmed: {len(cross_device_uids)}")

# Step 2: Build a per-user device journey summary
device_journey = (
    df_session[df_session['user_id'].isin(cross_device_uids)]
    .sort_values(['user_id','timestamp'])
    .groupby('user_id')
    .agg(
        first_device  = ('device',    'first'),   # first touch
        last_device   = ('device',    'last'),    # last touch
        converted     = ('event',     lambda x: 'purchase' in x.values),
        total_events  = ('event',     'count'),
        session_count = ('clean_session_id', 'nunique')
    )
    .reset_index()
)

print("\n--- Cross-Device User Journey Summary ---")
print(device_journey.head(10).to_string(index=False))

print(f"\nCross-device users who converted : {device_journey['converted'].sum()}")
print(f"Cross-device users who did not   : {(~device_journey['converted']).sum()}")

# Step 3: Single-device users for comparison baseline
single_device_uids = (
    df_session.groupby('user_id')['device']
    .nunique()
    .reset_index()
    .query('device == 1')
    ['user_id']
    .tolist()
)

single_journey = (
    df_session[df_session['user_id'].isin(single_device_uids)]
    .groupby('user_id')
    .agg(
        device        = ('device', 'first'),
        converted     = ('event',  lambda x: 'purchase' in x.values)
    )
    .reset_index()
)

# ============================================================
# Step 4: Calculate TRUE conversion rate per device
#         accounting for cross-device behavior
# ============================================================

print("\n" + "=" * 65)
print("  ATTRIBUTION MODEL COMPARISON")
print("=" * 65)

# --- NAIVE (broken) view ---
# Just count conversions by the device on the purchase event
purchase_events = df_session[df_session['event'] == 'purchase'].copy()
naive_conv = purchase_events['device'].value_counts().reset_index()
naive_conv.columns = ['device', 'naive_purchase_count']

all_users_by_device = df_session.groupby('device')['user_id'].nunique().reset_index()
all_users_by_device.columns = ['device', 'total_users']

naive_view = all_users_by_device.merge(naive_conv, on='device', how='left').fillna(0)
naive_view['naive_conv_rate'] = (
    naive_view['naive_purchase_count'] / naive_view['total_users']
).round(3)

print("\n  NAIVE VIEW (broken — ignores device switching):")
print(naive_view.to_string(index=False))

# --- FIRST TOUCH Attribution ---
first_touch_conv = (
    device_journey[device_journey['converted']]
    .groupby('first_device')
    .size()
    .reset_index(name='first_touch_conversions')
    .rename(columns={'first_device': 'device'})
)

# Add single device conversions
single_conv = (
    single_journey[single_journey['converted']]
    .groupby('device')
    .size()
    .reset_index(name='single_conversions')
)

first_touch_total = (
    first_touch_conv
    .merge(single_conv, on='device', how='outer')
    .fillna(0)
)
first_touch_total['total_first_touch'] = (
    first_touch_total['first_touch_conversions'] +
    first_touch_total['single_conversions']
)

print("\n  FIRST TOUCH ATTRIBUTION:")
print(first_touch_total[['device','first_touch_conversions',
                          'single_conversions',
                          'total_first_touch']].to_string(index=False))

# --- LAST TOUCH Attribution ---
last_touch_conv = (
    device_journey[device_journey['converted']]
    .groupby('last_device')
    .size()
    .reset_index(name='last_touch_conversions')
    .rename(columns={'last_device': 'device'})
)

last_touch_total = (
    last_touch_conv
    .merge(single_conv, on='device', how='outer')
    .fillna(0)
)
last_touch_total['total_last_touch'] = (
    last_touch_total['last_touch_conversions'] +
    last_touch_total['single_conversions']
)

print("\n  LAST TOUCH ATTRIBUTION:")
print(last_touch_total[['device','last_touch_conversions',
                         'single_conversions',
                         'total_last_touch']].to_string(index=False))

# --- LINEAR Attribution ---
print("\n  LINEAR ATTRIBUTION (50/50 split for cross-device):")
linear_records = []
for _, row in device_journey[device_journey['converted']].iterrows():
    if row['first_device'] != row['last_device']:
        linear_records.append({'device': row['first_device'], 'credit': 0.5})
        linear_records.append({'device': row['last_device'],  'credit': 0.5})
    else:
        linear_records.append({'device': row['first_device'], 'credit': 1.0})

linear_df = (
    pd.DataFrame(linear_records)
    .groupby('device')['credit']
    .sum()
    .reset_index()
    .rename(columns={'credit': 'linear_credit'})
)

# Add single device conversions
for _, row in single_journey[single_journey['converted']].iterrows():
    device = row['device']
    if device in linear_df['device'].values:
        linear_df.loc[linear_df['device'] == device, 'linear_credit'] += 1.0
    else:
        linear_df = pd.concat([
            linear_df,
            pd.DataFrame([{'device': device, 'linear_credit': 1.0}])
        ])

linear_df['linear_credit'] = linear_df['linear_credit'].round(1)
print(linear_df.to_string(index=False))

# ============================================================
# Step 5: The headline finding — mobile true drop-off
# ============================================================
print("\n" + "=" * 65)
print("  MOBILE DROP-OFF — NAIVE vs ATTRIBUTED")
print("=" * 65)

mobile_users = df_session[
    df_session['device'] == 'Mobile_App'
]['user_id'].nunique()

mobile_conversions_naive = purchase_events[
    purchase_events['device'] == 'Mobile_App'
]['user_id'].nunique()

mobile_conversions_first_touch = first_touch_total[
    first_touch_total['device'] == 'Mobile_App'
]['total_first_touch'].values[0] if 'Mobile_App' in first_touch_total['device'].values else 0

print(f"\n  Mobile users (touched mobile at any point) : {mobile_users}")
print(f"  Naive mobile conversions (last device=mobile): {int(mobile_conversions_naive)}")
print(f"  First-touch mobile conversions              : {int(mobile_conversions_first_touch)}")
print(f"\n  Naive mobile conversion rate    : "
      f"{mobile_conversions_naive/mobile_users:.1%}")
print(f"  First-touch mobile conv rate    : "
      f"{mobile_conversions_first_touch/mobile_users:.1%}")
print(f"\n  The gap between these two rates is what mobile")
print(f"  was being UNDERCREDITED by in your naive funnel.")
print("=" * 65)

Cross-device users confirmed: 34

--- Cross-Device User Journey Summary ---
  user_id first_device last_device  converted  total_events  session_count
user_0015       Tablet      Tablet       True             8              2
user_0018   Mobile_App  Mobile_App      False             4              2
user_0042  Web_Desktop Web_Desktop       True             5              2
user_0048  Web_Desktop Web_Desktop       True             5              2
user_0099  Web_Desktop Web_Desktop       True             5              2
user_0118       Tablet      Tablet      False             4              2
user_0122       Tablet      Tablet       True             5              2
user_0154       Tablet  Mobile_App      False             3              2
user_0174   Mobile_App Web_Desktop      False             4              2
user_0175  Web_Desktop  Mobile_App      False             3              2

Cross-device users who converted : 18
Cross-device users who did not   : 16

  ATTRIBUTION MODEL C

In [18]:
# ============================================================
# CORRECTED STAGE 4 — True Cross-Device Attribution
# ============================================================

# Step 1: Redefine cross-device users correctly
# A TRUE cross-device user must have:
# - More than one UNIQUE device value across ALL their events
# - Not just more than one session

true_cross_device = (
    df_session.groupby('user_id')['device']
    .nunique()
    .reset_index()
    .rename(columns={'device': 'unique_devices'})
)

# Split into true cross-device vs single-device populations
true_cross_uids = true_cross_device[
    true_cross_device['unique_devices'] > 1
]['user_id'].tolist()

true_single_uids = true_cross_device[
    true_cross_device['unique_devices'] == 1
]['user_id'].tolist()

print("=" * 55)
print("  CORRECTED POPULATION SPLIT")
print("=" * 55)
print(f"  True cross-device users  : {len(true_cross_uids)}")
print(f"  True single-device users : {len(true_single_uids)}")
print("=" * 55)

# Step 2: Rebuild device journey for TRUE cross-device users only
true_cross_journey = (
    df_session[df_session['user_id'].isin(true_cross_uids)]
    .sort_values(['user_id', 'timestamp'])
    .groupby('user_id')
    .agg(
        first_device = ('device', 'first'),
        last_device  = ('device', 'last'),
        all_devices  = ('device', lambda x: list(x.unique())),
        converted    = ('event',  lambda x: 'purchase' in x.values),
        total_events = ('event',  'count')
    )
    .reset_index()
)

print("\n--- TRUE Cross-Device Journeys ---")
print(true_cross_journey.to_string(index=False))

print(f"\nTrue cross-device converters     : {true_cross_journey['converted'].sum()}")
print(f"True cross-device non-converters : {(~true_cross_journey['converted']).sum()}")

# Step 3: Single device journey summary
true_single_journey = (
    df_session[df_session['user_id'].isin(true_single_uids)]
    .groupby('user_id')
    .agg(
        device    = ('device', 'first'),
        converted = ('event',  lambda x: 'purchase' in x.values)
    )
    .reset_index()
)

# ============================================================
# Step 4: Build all three attribution models CORRECTLY
# ============================================================

# Single device conversions per device (baseline — same for all models)
single_conv_by_device = (
    true_single_journey[true_single_journey['converted']]
    .groupby('device')
    .size()
    .reset_index(name='single_conversions')
)

# All devices present in data
all_devices = df_session['device'].unique()
device_index = pd.DataFrame({'device': all_devices})

# ── FIRST TOUCH ──
first_touch = (
    true_cross_journey[true_cross_journey['converted']]
    .groupby('first_device')
    .size()
    .reset_index(name='cross_conversions')
    .rename(columns={'first_device': 'device'})
)

first_touch_final = (
    device_index
    .merge(first_touch,       on='device', how='left')
    .merge(single_conv_by_device, on='device', how='left')
    .fillna(0)
)
first_touch_final['total'] = (
    first_touch_final['cross_conversions'] +
    first_touch_final['single_conversions']
)

# ── LAST TOUCH ──
last_touch = (
    true_cross_journey[true_cross_journey['converted']]
    .groupby('last_device')
    .size()
    .reset_index(name='cross_conversions')
    .rename(columns={'last_device': 'device'})
)

last_touch_final = (
    device_index
    .merge(last_touch,        on='device', how='left')
    .merge(single_conv_by_device, on='device', how='left')
    .fillna(0)
)
last_touch_final['total'] = (
    last_touch_final['cross_conversions'] +
    last_touch_final['single_conversions']
)

# ── LINEAR (50/50 for cross-device converters) ──
linear_records = []

# Cross-device converters get split credit
for _, row in true_cross_journey[true_cross_journey['converted']].iterrows():
    if row['first_device'] != row['last_device']:
        linear_records.append({'device': row['first_device'], 'credit': 0.5})
        linear_records.append({'device': row['last_device'],  'credit': 0.5})
    else:
        linear_records.append({'device': row['first_device'], 'credit': 1.0})

# Single device converters get full credit
for _, row in true_single_journey[true_single_journey['converted']].iterrows():
    linear_records.append({'device': row['device'], 'credit': 1.0})

linear_final = (
    pd.DataFrame(linear_records)
    .groupby('device')['credit']
    .sum()
    .reset_index()
    .rename(columns={'credit': 'linear_credit'})
)
linear_final['linear_credit'] = linear_final['linear_credit'].round(1)

# ============================================================
# Step 5: Full comparison table
# ============================================================
attribution_comparison = (
    device_index
    .merge(first_touch_final[['device','total']].rename(
        columns={'total': 'first_touch'}), on='device', how='left')
    .merge(last_touch_final[['device','total']].rename(
        columns={'total': 'last_touch'}), on='device', how='left')
    .merge(linear_final, on='device', how='left')
    .fillna(0)
)

# Add total users per device for conversion rate context
users_per_device = (
    df_session.groupby('device')['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'total_users'})
)
attribution_comparison = attribution_comparison.merge(
    users_per_device, on='device'
)

attribution_comparison['first_touch_rate'] = (
    attribution_comparison['first_touch'] /
    attribution_comparison['total_users']
).round(3)

attribution_comparison['last_touch_rate'] = (
    attribution_comparison['last_touch'] /
    attribution_comparison['total_users']
).round(3)

attribution_comparison['linear_rate'] = (
    attribution_comparison['linear_credit'] /
    attribution_comparison['total_users']
).round(3)

print("\n" + "=" * 75)
print("  CORRECTED ATTRIBUTION COMPARISON — All Three Models")
print("=" * 75)
print(attribution_comparison[[
    'device', 'total_users',
    'first_touch', 'first_touch_rate',
    'last_touch',  'last_touch_rate',
    'linear_credit','linear_rate'
]].to_string(index=False))
print("=" * 75)

# ============================================================
# Step 6: The headline finding
# ============================================================
print("\n" + "=" * 75)
print("  HEADLINE FINDING — Which device DISCOVERS vs CLOSES?")
print("=" * 75)

for _, row in attribution_comparison.iterrows():
    ft = row['first_touch']
    lt = row['last_touch']
    device = row['device']

    if ft > lt:
        role = "DISCOVERY ENGINE  — starts more journeys than it closes"
    elif lt > ft:
        role = "CONVERSION ENGINE — closes more journeys than it starts"
    else:
        role = "BALANCED          — equal discovery and conversion weight"

    print(f"\n  {device}")
    print(f"    First touch conversions : {int(ft)}")
    print(f"    Last touch conversions  : {int(lt)}")
    print(f"    Role                    : {role}")

print("\n" + "=" * 75)


  CORRECTED POPULATION SPLIT
  True cross-device users  : 34
  True single-device users : 434

--- TRUE Cross-Device Journeys ---
  user_id first_device last_device               all_devices  converted  total_events
user_0015       Tablet      Tablet      [Tablet, Mobile_App]       True             8
user_0018   Mobile_App  Mobile_App [Mobile_App, Web_Desktop]      False             4
user_0042  Web_Desktop Web_Desktop [Web_Desktop, Mobile_App]       True             5
user_0048  Web_Desktop Web_Desktop [Web_Desktop, Mobile_App]       True             5
user_0099  Web_Desktop Web_Desktop [Web_Desktop, Mobile_App]       True             5
user_0118       Tablet      Tablet      [Tablet, Mobile_App]      False             4
user_0122       Tablet      Tablet      [Tablet, Mobile_App]       True             5
user_0154       Tablet  Mobile_App      [Tablet, Mobile_App]      False             3
user_0174   Mobile_App Web_Desktop [Mobile_App, Web_Desktop]      False             4
user_0175 

In [19]:
# ============================================================
# DIAGNOSTIC — understand the actual device switch patterns
# ============================================================

print("=" * 65)
print("  DEVICE SWITCH PATTERN DIAGNOSTIC")
print("=" * 65)

for uid in true_cross_uids:
    user_events = (
        df_session[df_session['user_id'] == uid]
        .sort_values('timestamp')
        [['event', 'device', 'timestamp']]
    )

    devices_in_order = user_events['device'].tolist()
    events_in_order  = user_events['event'].tolist()

    # Find where the device actually changed
    switch_indices = []
    for i in range(1, len(devices_in_order)):
        if devices_in_order[i] != devices_in_order[i-1]:
            switch_indices.append(i)

    if switch_indices:
        switch_event = events_in_order[switch_indices[0]]
        from_device  = devices_in_order[switch_indices[0] - 1]
        to_device    = devices_in_order[switch_indices[0]]
        converted    = 'purchase' in events_in_order

        print(f"\n  {uid}")
        print(f"    Switch at event : {switch_event}")
        print(f"    From device     : {from_device}")
        print(f"    To device       : {to_device}")
        print(f"    Converted       : {converted}")
        print(f"    Full journey    : {list(zip(events_in_order, devices_in_order))}")

  DEVICE SWITCH PATTERN DIAGNOSTIC

  user_0015
    Switch at event : add_to_cart
    From device     : Tablet
    To device       : Mobile_App
    Converted       : True
    Full journey    : [('home', 'Tablet'), ('product_view', 'Tablet'), ('add_to_cart', 'Mobile_App'), ('home', 'Mobile_App'), ('product_view', 'Mobile_App'), ('add_to_cart', 'Mobile_App'), ('checkout_click', 'Tablet'), ('purchase', 'Tablet')]

  user_0018
    Switch at event : add_to_cart
    From device     : Mobile_App
    To device       : Web_Desktop
    Converted       : False
    Full journey    : [('home', 'Mobile_App'), ('product_view', 'Mobile_App'), ('add_to_cart', 'Web_Desktop'), ('checkout_click', 'Mobile_App')]

  user_0042
    Switch at event : checkout_click
    From device     : Web_Desktop
    To device       : Mobile_App
    Converted       : True
    Full journey    : [('home', 'Web_Desktop'), ('product_view', 'Web_Desktop'), ('add_to_cart', 'Web_Desktop'), ('checkout_click', 'Mobile_App'), ('purcha

In [20]:
# ============================================================
# POSITION-BASED ATTRIBUTION — The correct model for
# mid-funnel device switching behavior
# ============================================================
#
# Credit rules:
#   Primary device (owned first AND last touch) = 80% credit
#   Middle device (appeared only mid-funnel)    = 20% credit
#
#   For users who genuinely start and end on different devices:
#   First device  = 40% credit
#   Middle device = 20% credit (if exists)
#   Last device   = 40% credit
# ============================================================

def get_device_credits(journey_tuples, converted):
    """
    Input : list of (event, device) tuples in chronological order
    Output: dict of {device: credit_share} for this user
    """
    if not converted:
        return {}   # only attribute converted journeys

    devices_in_order = [d for _, d in journey_tuples]
    unique_devices   = list(dict.fromkeys(devices_in_order))  # ordered, deduplicated

    first_device = devices_in_order[0]
    last_device  = devices_in_order[-1]

    if len(unique_devices) == 1:
        # Single device journey — full credit
        return {first_device: 1.0}

    middle_devices = [d for d in unique_devices
                      if d != first_device and d != last_device]

    if first_device == last_device:
        # PRIMARY device owns first and last touch → 80%
        # Middle device(s) share the remaining 20%
        credits = {first_device: 0.80}
        if middle_devices:
            share = 0.20 / len(middle_devices)
            for md in middle_devices:
                credits[md] = credits.get(md, 0) + share
        else:
            # Only one other device appeared and it came back
            # Give middle device 20% anyway
            mid = [d for d in unique_devices if d != first_device][0]
            credits[mid] = 0.20
        return credits

    else:
        # Genuine start-to-end device switch
        # First = 40%, Last = 40%, Middle = 20% split
        credits = {first_device: 0.40, last_device: 0.40}
        if middle_devices:
            share = 0.20 / len(middle_devices)
            for md in middle_devices:
                if md in credits:
                    credits[md] += share
                else:
                    credits[md] = share
        else:
            # No middle — split 50/50
            credits = {first_device: 0.50, last_device: 0.50}
        return credits


# ============================================================
# Apply position-based credits to all cross-device converters
# ============================================================
position_records = []

for uid in true_cross_uids:
    user_events = (
        df_session[df_session['user_id'] == uid]
        .sort_values('timestamp')
    )
    journey  = list(zip(user_events['event'], user_events['device']))
    converted = 'purchase' in user_events['event'].values

    credits = get_device_credits(journey, converted)
    for device, credit in credits.items():
        position_records.append({
            'user_id': uid,
            'device':  device,
            'credit':  credit
        })

position_cross = (
    pd.DataFrame(position_records)
    .groupby('device')['credit']
    .sum()
    .reset_index()
    .rename(columns={'credit': 'position_credit'})
)

# Add single-device converter credits (always 1.0 each)
for _, row in true_single_journey[true_single_journey['converted']].iterrows():
    device = row['device']
    if device in position_cross['device'].values:
        position_cross.loc[
            position_cross['device'] == device, 'position_credit'
        ] += 1.0
    else:
        position_cross = pd.concat([
            position_cross,
            pd.DataFrame([{'device': device, 'position_credit': 1.0}])
        ]).reset_index(drop=True)

position_cross['position_credit'] = position_cross['position_credit'].round(2)

# ============================================================
# Build the final three-model comparison
# ============================================================

# First touch — credit goes to first device of each converter
first_records = []
for uid in true_cross_uids:
    user_events = (
        df_session[df_session['user_id'] == uid]
        .sort_values('timestamp')
    )
    converted = 'purchase' in user_events['event'].values
    if converted:
        first_records.append({
            'device': user_events['device'].iloc[0],
            'credit': 1.0
        })
for _, row in true_single_journey[true_single_journey['converted']].iterrows():
    first_records.append({'device': row['device'], 'credit': 1.0})

first_touch_df = (
    pd.DataFrame(first_records)
    .groupby('device')['credit']
    .sum()
    .reset_index()
    .rename(columns={'credit': 'first_touch'})
)

# Last touch — credit goes to last device of each converter
last_records = []
for uid in true_cross_uids:
    user_events = (
        df_session[df_session['user_id'] == uid]
        .sort_values('timestamp')
    )
    converted = 'purchase' in user_events['event'].values
    if converted:
        last_records.append({
            'device': user_events['device'].iloc[-1],
            'credit': 1.0
        })
for _, row in true_single_journey[true_single_journey['converted']].iterrows():
    last_records.append({'device': row['device'], 'credit': 1.0})

last_touch_df = (
    pd.DataFrame(last_records)
    .groupby('device')['credit']
    .sum()
    .reset_index()
    .rename(columns={'credit': 'last_touch'})
)

# ============================================================
# Final comparison table
# ============================================================
final_attribution = (
    device_index
    .merge(first_touch_df,  on='device', how='left')
    .merge(last_touch_df,   on='device', how='left')
    .merge(position_cross,  on='device', how='left')
    .merge(users_per_device, on='device', how='left')
    .fillna(0)
)

final_attribution['first_touch_rate']  = (
    final_attribution['first_touch']      / final_attribution['total_users']
).round(3)
final_attribution['last_touch_rate']   = (
    final_attribution['last_touch']       / final_attribution['total_users']
).round(3)
final_attribution['position_rate']     = (
    final_attribution['position_credit']  / final_attribution['total_users']
).round(3)

print("=" * 75)
print("  FINAL ATTRIBUTION — Three Models Side by Side")
print("=" * 75)
print(final_attribution[[
    'device','total_users',
    'first_touch','first_touch_rate',
    'last_touch', 'last_touch_rate',
    'position_credit','position_rate'
]].to_string(index=False))
print("=" * 75)

# ============================================================
# Discovery vs Conversion engine analysis
# ============================================================
print("\n" + "=" * 75)
print("  DEVICE ROLE ANALYSIS")
print("=" * 75)

for _, row in final_attribution.iterrows():
    ft   = row['first_touch']
    lt   = row['last_touch']
    pos  = row['position_credit']
    dev  = row['device']
    gap  = ft - lt

    if gap > 2:
        role = "DISCOVERY ENGINE  — starts more journeys than it closes"
    elif gap < -2:
        role = "CONVERSION ENGINE — closes more sales than it starts"
    else:
        role = "BALANCED          — roughly equal discovery and conversion"

    print(f"\n  {dev}")
    print(f"    First touch credit   : {ft:.1f}")
    print(f"    Last touch credit    : {lt:.1f}")
    print(f"    Position credit      : {pos:.1f}  ← most accurate view")
    print(f"    Discovery gap        : {gap:+.1f}  (first minus last)")
    print(f"    Role                 : {role}")

print("\n" + "=" * 75)

# ============================================================
# The mobile mid-funnel finding — the key business insight
# ============================================================
print("\n  MID-FUNNEL DEVICE INFLUENCE SUMMARY")
print("=" * 75)

mid_funnel_counts = {}
for uid in true_cross_uids:
    user_events = (
        df_session[df_session['user_id'] == uid]
        .sort_values('timestamp')
    )
    journey   = list(zip(user_events['event'], user_events['device']))
    converted = 'purchase' in user_events['event'].values
    if not converted:
        continue

    devices   = [d for _, d in journey]
    first_dev = devices[0]
    last_dev  = devices[-1]

    # Find devices that appeared but are neither first nor last
    middle = set(devices) - {first_dev, last_dev}
    for md in middle:
        mid_funnel_counts[md] = mid_funnel_counts.get(md, 0) + 1

    # If first == last, the middle device is the one that switched
    if first_dev == last_dev:
        other = [d for d in set(devices) if d != first_dev]
        for od in other:
            mid_funnel_counts[od] = mid_funnel_counts.get(od, 0) + 1

print("\n  Devices appearing as MID-FUNNEL touch in converted journeys:")
for device, count in sorted(mid_funnel_counts.items(),
                            key=lambda x: x[1], reverse=True):
    print(f"    {device:15s} : {count} converted journeys influenced")

print("\n  These devices received ZERO credit under First and Last Touch.")
print("  Position-Based model captures their true influence.")
print("=" * 75)


  FINAL ATTRIBUTION — Three Models Side by Side
     device  total_users  first_touch  first_touch_rate  last_touch  last_touch_rate  position_credit  position_rate
     Tablet          153         42.0             0.275        42.0            0.275             41.0          0.268
 Mobile_App          178         59.0             0.331        59.0            0.331             59.8          0.336
Web_Desktop          171         53.0             0.310        53.0            0.310             53.2          0.311

  DEVICE ROLE ANALYSIS

  Tablet
    First touch credit   : 42.0
    Last touch credit    : 42.0
    Position credit      : 41.0  ← most accurate view
    Discovery gap        : +0.0  (first minus last)
    Role                 : BALANCED          — roughly equal discovery and conversion

  Mobile_App
    First touch credit   : 59.0
    Last touch credit    : 59.0
    Position credit      : 59.8  ← most accurate view
    Discovery gap        : +0.0  (first minus last)
    Role  

In [21]:
# ============================================================
# STAGE 4 FINAL SUMMARY — The Business Translation
# ============================================================

total_conversions = int(true_single_journey['converted'].sum() +
                        true_cross_journey['converted'].sum())

mobile_last_touch    = int(final_attribution[
    final_attribution['device'] == 'Mobile_App']['last_touch'].values[0])
mobile_position      = float(final_attribution[
    final_attribution['device'] == 'Mobile_App']['position_credit'].values[0])
mobile_mid_influence = mid_funnel_counts.get('Mobile_App', 0)

web_last_touch       = int(final_attribution[
    final_attribution['device'] == 'Web_Desktop']['last_touch'].values[0])
web_position         = float(final_attribution[
    final_attribution['device'] == 'Web_Desktop']['position_credit'].values[0])
web_mid_influence    = mid_funnel_counts.get('Web_Desktop', 0)

print("=" * 65)
print("  STAGE 4 COMPLETE — EXECUTIVE SUMMARY")
print("=" * 65)

print(f"""
  TOTAL CONFIRMED CONVERSIONS     : {total_conversions}
  TRUE CROSS-DEVICE USERS         : {len(true_cross_uids)}
  CROSS-DEVICE CONVERTERS         : {true_cross_journey['converted'].sum()}

  ── MOBILE APP ──────────────────────────────────────
  Last-touch conversions credited : {mobile_last_touch}
  Position-based credit           : {mobile_position:.1f}
  Mid-funnel influence (hidden)   : {mobile_mid_influence} journeys
  Hidden influence rate           : {mobile_mid_influence/total_conversions:.1%} of all conversions

  What this means:
  Mobile influenced {mobile_last_touch + mobile_mid_influence} total conversions
  but standard attribution only credited it for {mobile_last_touch}.
  {mobile_mid_influence} conversions ({mobile_mid_influence/total_conversions:.1%}) were
  invisible under first-touch and last-touch models.

  ── WEB DESKTOP ─────────────────────────────────────
  Last-touch conversions credited : {web_last_touch}
  Position-based credit           : {web_position:.1f}
  Mid-funnel influence (hidden)   : {web_mid_influence} journeys

  ── THE STRUCTURAL FINDING ──────────────────────────
  Mobile_App is a mid-funnel authentication device.
  Users switch TO mobile at cart or checkout —
  likely to use saved payment credentials or
  biometric authentication (Face ID, fingerprint).
  They then return to their primary device to confirm.

  Under naive attribution: this behavior is invisible.
  Under position-based:    mobile gets partial credit.
  True mobile influence    = {mobile_last_touch} direct + {mobile_mid_influence} assisted
                           = {mobile_last_touch + mobile_mid_influence} total touched conversions
                           = {(mobile_last_touch + mobile_mid_influence)/total_conversions:.1%} of all conversions
""")
print("=" * 65)

# ============================================================
# FULL PIPELINE SUMMARY — all four findings together
# ============================================================
print("\n" + "=" * 65)
print("  COMPLETE ANALYSIS — ALL FINDINGS")
print("=" * 65)
print(f"""
  RAW DATA
  ─────────────────────────────────────────────
  Total raw events          : 1,711
  Unique users              : 468
  Reported purchase events  : 154

  FINDING 1 — Orphaned Purchases
  ─────────────────────────────────────────────
  Orphaned purchase users   : 13 (8.4% of purchases)
  Naive checkout conv rate  : 83.7%
  Strict checkout conv rate : 76.6%
  Inflation                 : 7.1 percentage points
  Action                    : Flag to data engineering
                              Audit server-side tracking

  FINDING 2 — Duplicate Event Inflation
  ─────────────────────────────────────────────
  Duplicate events removed  : 90
  All from                  : checkout_click only
  Raw checkout volume       : 274 events
  True checkout volume      : 184 unique user events
  Overcount                 : 49%
  Action                    : Add debounce logic to
                              checkout button SDK

  FINDING 3 — Session Boundary Detection
  ─────────────────────────────────────────────
  Users with split sessions : 34
  All caused by             : Cross-device switches
  Avg gap at switch point   : 602+ minutes
  Action                    : Implement 30-min session
                              timeout in ETL pipeline

  FINDING 4 — Attribution Model Limitations
  ─────────────────────────────────────────────
  True cross-device users   : 34 (7.3% of all users)
  Mobile mid-funnel touches : {mobile_mid_influence} converted journeys
  Web mid-funnel touches    : {web_mid_influence} converted journeys
  Hidden mobile influence   : {mobile_mid_influence/total_conversions:.1%} of conversions
  Standard models missed    : {mobile_mid_influence + web_mid_influence} influenced journeys
  Action                    : Replace last-touch with
                              position-based attribution
                              in marketing dashboards
""")
print("=" * 65)

  STAGE 4 COMPLETE — EXECUTIVE SUMMARY

  TOTAL CONFIRMED CONVERSIONS     : 154
  TRUE CROSS-DEVICE USERS         : 34
  CROSS-DEVICE CONVERTERS         : 18

  ── MOBILE APP ──────────────────────────────────────
  Last-touch conversions credited : 59
  Position-based credit           : 59.8
  Mid-funnel influence (hidden)   : 22 journeys
  Hidden influence rate           : 14.3% of all conversions

  What this means:
  Mobile influenced 81 total conversions
  but standard attribution only credited it for 59.
  22 conversions (14.3%) were
  invisible under first-touch and last-touch models.

  ── WEB DESKTOP ─────────────────────────────────────
  Last-touch conversions credited : 53
  Position-based credit           : 53.2
  Mid-funnel influence (hidden)   : 14 journeys

  ── THE STRUCTURAL FINDING ──────────────────────────
  Mobile_App is a mid-funnel authentication device.
  Users switch TO mobile at cart or checkout —
  likely to use saved payment credentials or
  biometric authe

In [22]:
import os

# Create all the folders in one go
os.makedirs('data/raw',        exist_ok=True)
os.makedirs('data/processed',  exist_ok=True)
os.makedirs('notebooks',       exist_ok=True)
os.makedirs('src',             exist_ok=True)
os.makedirs('outputs',         exist_ok=True)

print("Folders created:")
for folder in ['data/raw', 'data/processed', 'notebooks', 'src', 'outputs']:
    print(f"  {folder}")

Folders created:
  data/raw
  data/processed
  notebooks
  src
  outputs


## Summary of Findings

| Finding | Detail |
|---|---|
| Orphaned purchases | 13 users inflated checkout conv by 7.1pts |
| Duplicate events | 90 phantom checkout clicks removed (49% overcount) |
| Session boundaries | 34 cross-device sessions misread as drop-offs |
| Hidden mobile influence | 22 conversions invisible under standard attribution |

In [23]:
# ============================================================
# EXPORT — Save all data layers for the repository
# ============================================================

# Raw dataset — the intentionally dirty original
df.to_csv('data/raw/events_log_raw.csv', index=False)

# Deduplicated dataset
df_deduped.to_csv('data/processed/events_deduped.csv', index=False)

# Sessionized dataset — the final clean working layer
df_session.to_csv('data/processed/events_sessionized.csv', index=False)

print("Data exported successfully.")
print(f"  Raw events      : {len(df):,} rows")
print(f"  Deduped events  : {len(df_deduped):,} rows")
print(f"  Sessionized     : {len(df_session):,} rows")

Data exported successfully.
  Raw events      : 1,711 rows
  Deduped events  : 1,621 rows
  Sessionized     : 1,621 rows


In [28]:
import os

# Check exactly what we have
print("FILES IN /content/data/raw:")
for f in os.listdir('/content/data/raw'):
    print(f"  {f}")

print("\nFILES IN /content/data/processed:")
for f in os.listdir('/content/data/processed'):
    print(f"  {f}")

print("\nFILES IN /content/src:")
for f in os.listdir('/content/src') if os.path.exists('/content/src') else []:
    print(f"  {f}")

print("\nFILES IN /content (root):")
for f in os.listdir('/content'):
    if os.path.isfile(f'/content/{f}'):
        print(f"  {f}")

FILES IN /content/data/raw:
  events_log_raw.csv

FILES IN /content/data/processed:
  events_deduped.csv
  events_sessionized.csv

FILES IN /content/src:

FILES IN /content (root):
  funnel-analysis-edge-cases.zip


In [30]:
import os
import shutil
from google.colab import files

# ============================================================
# CREATE MISSING FILES
# ============================================================

# requirements.txt
with open('/content/requirements.txt', 'w') as f:
    f.write("""pandas==2.1.0
numpy==1.24.0
matplotlib==3.7.0
seaborn==0.12.0
jupyter==1.0.0
""")

# funnel_utils.py
with open('/content/src/funnel_utils.py', 'w') as f:
    f.write('''import pandas as pd
import numpy as np


def build_naive_funnel(df, funnel_steps):
    """
    Naive funnel — unique users per step, no sequence enforcement.
    Warning: intentionally overcounts. Use to demonstrate inflation only.
    """
    results = []
    for step in funnel_steps:
        unique_users = df[df["event"] == step]["user_id"].nunique()
        results.append({"step": step, "unique_users": unique_users})
    result_df = pd.DataFrame(results)
    result_df["prev_users"] = result_df["unique_users"].shift(1)
    result_df["conversion_rate"] = (
        result_df["unique_users"] / result_df["prev_users"]
    ).round(3)
    top = result_df.loc[0, "unique_users"]
    result_df["overall_conv"] = (result_df["unique_users"] / top).round(3)
    return result_df


def build_strict_funnel(df, funnel_steps):
    """
    Strict sequence funnel — enforces chronological step order.
    User must complete step N before step N+1 to qualify.
    """
    first_touch = (
        df[df["event"].isin(funnel_steps)]
        .groupby(["user_id", "event"])["timestamp"]
        .min()
        .reset_index()
        .rename(columns={"timestamp": "first_ts"})
    )
    pivot_df = first_touch.pivot_table(
        index="user_id", columns="event",
        values="first_ts", aggfunc="min"
    )
    pivot_df = pivot_df[funnel_steps]
    pivot_df.columns.name = None
    strict_funnel = []
    qualifying = pivot_df.copy()
    for i, step in enumerate(funnel_steps):
        if i == 0:
            mask = qualifying[step].notna()
        else:
            prev = funnel_steps[i - 1]
            mask = (
                qualifying[step].notna() &
                (qualifying[step] > qualifying[prev])
            )
        qualifying = qualifying[mask]
        strict_funnel.append({"step": step, "strict_users": len(qualifying)})
    result_df = pd.DataFrame(strict_funnel)
    result_df["prev_users"] = result_df["strict_users"].shift(1)
    result_df["conversion_rate"] = (
        result_df["strict_users"] / result_df["prev_users"]
    ).round(3)
    top = result_df.loc[0, "strict_users"]
    result_df["overall_conv"] = (result_df["strict_users"] / top).round(3)
    return result_df


def deduplicate_events(df, event_col="event", time_col="timestamp",
                       user_col="user_id", window_seconds=5):
    """
    Remove duplicate events fired within a short time window.
    Targets SDK retries, button double-taps, race conditions.
    """
    df_sorted = df.sort_values([user_col, time_col]).reset_index(drop=True)
    df_sorted["_prev_event"] = df_sorted.groupby(user_col)[event_col].shift(1)
    df_sorted["_prev_ts"]    = df_sorted.groupby(user_col)[time_col].shift(1)
    df_sorted["_time_diff"]  = (
        df_sorted[time_col] - df_sorted["_prev_ts"]
    ).dt.total_seconds()
    is_duplicate = (
        (df_sorted[event_col] == df_sorted["_prev_event"]) &
        (df_sorted["_time_diff"] <= window_seconds)
    )
    df_clean = df_sorted[~is_duplicate].copy()
    df_clean = df_clean.drop(columns=["_prev_event", "_prev_ts", "_time_diff"])
    return df_clean


def sessionize(df, user_col="user_id", time_col="timestamp",
               timeout_minutes=30):
    """
    Assign session labels using inactivity timeout.
    New session starts when user is idle beyond timeout_minutes.
    """
    df_out = df.sort_values([user_col, time_col]).reset_index(drop=True).copy()
    df_out["_prev_ts"]        = df_out.groupby(user_col)[time_col].shift(1)
    df_out["gap_minutes"]     = (
        df_out[time_col] - df_out["_prev_ts"]
    ).dt.total_seconds() / 60
    df_out["_is_new_session"] = (
        df_out["gap_minutes"].isna() |
        (df_out["gap_minutes"] > timeout_minutes)
    )
    df_out["_session_num"]    = (
        df_out.groupby(user_col)["_is_new_session"].cumsum().astype(int)
    )
    df_out["clean_session_id"] = (
        df_out[user_col] + "_session_" + df_out["_session_num"].astype(str)
    )
    df_out = df_out.drop(
        columns=["_prev_ts", "_is_new_session", "_session_num"]
    )
    return df_out


def get_device_credits(journey_tuples, converted):
    """
    Position-based attribution for cross-device journeys.
    Primary device (first=last): 80% credit.
    Mid-funnel device:           20% credit.
    True start-to-end switch:    40% first, 20% middle, 40% last.
    """
    if not converted:
        return {}
    devices      = [d for _, d in journey_tuples]
    unique_devs  = list(dict.fromkeys(devices))
    first_device = devices[0]
    last_device  = devices[-1]
    if len(unique_devs) == 1:
        return {first_device: 1.0}
    other = [d for d in unique_devs if d != first_device]
    if first_device == last_device:
        credits = {first_device: 0.80}
        share   = 0.20 / len(other)
        for d in other:
            credits[d] = credits.get(d, 0) + share
        return credits
    else:
        middle  = [d for d in unique_devs
                   if d != first_device and d != last_device]
        credits = {first_device: 0.40, last_device: 0.40}
        if middle:
            share = 0.20 / len(middle)
            for d in middle:
                credits[d] = credits.get(d, 0) + share
        else:
            credits = {first_device: 0.50, last_device: 0.50}
        return credits
''')

print("✓ requirements.txt created")
print("✓ funnel_utils.py created")

# ============================================================
# VERIFY ALL 5 FILES EXIST
# ============================================================
all_files = [
    '/content/data/raw/events_log_raw.csv',
    '/content/data/processed/events_deduped.csv',
    '/content/data/processed/events_sessionized.csv',
    '/content/requirements.txt',
    '/content/src/funnel_utils.py'
]

print("\nFILE CHECK:")
all_good = True
for path in all_files:
    exists = os.path.exists(path)
    status = "✓ EXISTS" if exists else "✗ MISSING"
    if not exists:
        all_good = False
    print(f"  {status}  {path.replace('/content/','')}")

# ============================================================
# ZIP AND DOWNLOAD
# ============================================================
if all_good:
    # Remove old broken zip first
    if os.path.exists('/content/funnel-analysis-edge-cases.zip'):
        os.remove('/content/funnel-analysis-edge-cases.zip')

    # Build clean export folder
    if os.path.exists('/content/project_export'):
        shutil.rmtree('/content/project_export')

    os.makedirs('/content/project_export/data/raw',       exist_ok=True)
    os.makedirs('/content/project_export/data/processed',  exist_ok=True)
    os.makedirs('/content/project_export/src',             exist_ok=True)
    os.makedirs('/content/project_export/notebooks',       exist_ok=True)
    os.makedirs('/content/project_export/outputs',         exist_ok=True)

    # Copy all files into export folder
    shutil.copy('/content/data/raw/events_log_raw.csv',
                '/content/project_export/data/raw/')
    shutil.copy('/content/data/processed/events_deduped.csv',
                '/content/project_export/data/processed/')
    shutil.copy('/content/data/processed/events_sessionized.csv',
                '/content/project_export/data/processed/')
    shutil.copy('/content/requirements.txt',
                '/content/project_export/')
    shutil.copy('/content/src/funnel_utils.py',
                '/content/project_export/src/')

    # Zip and download
    shutil.make_archive(
        '/content/funnel-analysis-edge-cases', 'zip',
        '/content/project_export'
    )
    print("\n✓ Zip created — downloading now...")
    files.download('/content/funnel-analysis-edge-cases.zip')

else:
    print("\n✗ Some files missing — fix errors above before zipping.")

✓ requirements.txt created
✓ funnel_utils.py created

FILE CHECK:
  ✓ EXISTS  data/raw/events_log_raw.csv
  ✓ EXISTS  data/processed/events_deduped.csv
  ✓ EXISTS  data/processed/events_sessionized.csv
  ✓ EXISTS  requirements.txt
  ✓ EXISTS  src/funnel_utils.py

✓ Zip created — downloading now...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>